In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/phuc23020636/output-qwen05/__huggingface_repos__.json
/kaggle/input/datasets/phuc23020636/output-qwen05/outputs/zeroshot_qwen2_5_0_5b_chatmask.csv
/kaggle/input/datasets/phuc23020636/output-qwen05/outputs/zeroshot_qwen2_5_0_5b_chatmask_eval.csv
/kaggle/input/datasets/phuc23020636/output-qwen05/outputs/finetune_qwen2_5_0_5b_chatmask.csv
/kaggle/input/datasets/phuc23020636/output-qwen05/outputs/qwen_metrics_chatmask.csv
/kaggle/input/datasets/phuc23020636/output-qwen05/outputs/finetune_qwen2_5_0_5b_chatmask_eval.csv
/kaggle/input/datasets/phuc23020636/output-qwen05/checkpoints/qwen2_5_0_5b_lora_chatmask/adapter_model.safetensors
/kaggle/input/datasets/phuc23020636/output-qwen05/checkpoints/qwen2_5_0_5b_lora_chatmask/training_args.bin
/kaggle/input/datasets/phuc23020636/output-qwen05/checkpoints/qwen2_5_0_5b_lora_chatmask/adapter_config.json
/kaggle/input/datasets/phuc23020636/output-qwen05/checkpoints/qwen2_5_0_5b_lora_chatmask/README.md
/kaggle/input/datasets/phuc

In [2]:
import pandas as pd

TEST_PATH = "/kaggle/input/datasets/lhuythc/test-official/test-00000-of-00001 (1).parquet"

def read_any(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path.endswith(".csv"):
        return pd.read_csv(path)
    elif path.endswith(".json"):
        return pd.read_json(path)
    elif path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    else:
        raise ValueError(f"Unsupported file type: {path}")

raw_test_df = read_any(TEST_PATH)

print(raw_test_df.shape)
print(raw_test_df.columns)
raw_test_df.head()

(1344, 2)
Index(['article', 'summary'], dtype='object')


,article,summary
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."
3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Du khách Anh Monisha Rajesh bắt đầu hành trình...
4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Cồn Én là một điểm đến du lịch nằm giữa sông T...


In [3]:
def clean_test_df(df, text_col="article", summary_col="summary"):
    df = df.copy()

    if text_col not in df.columns:
        raise ValueError(f"Không tìm thấy cột {text_col}. Columns: {df.columns.tolist()}")

    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].str.len() > 50].reset_index(drop=True)

    if summary_col in df.columns:
        df[summary_col] = df[summary_col].astype(str).str.strip()
        df = df.rename(columns={text_col: "article", summary_col: "summary"})
        return df[["article", "summary"]].reset_index(drop=True)
    else:
        df = df.rename(columns={text_col: "article"})
        return df[["article"]].reset_index(drop=True)

official_test_df = clean_test_df(raw_test_df)

print(official_test_df.shape)
official_test_df.head()

(1344, 2)


,article,summary
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."
3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Du khách Anh Monisha Rajesh bắt đầu hành trình...
4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Cồn Én là một điểm đến du lịch nằm giữa sông T...


In [4]:
ADAPTER_DIR = "/kaggle/input/datasets/lhuythc/output-bartpho/checkpoints/bartpho_word_lora_correct"

In [5]:
import os

ADAPTER_DIR = "/kaggle/input/datasets/lhuythc/output-bartpho/checkpoints/bartpho_word_lora_correct"

print(os.path.exists(ADAPTER_DIR))
print(os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")))

True
True


In [6]:
!pip install -q transformers peft sentencepiece underthesea evaluate rouge_score accelerate

import os
import gc
import time
import torch
import pandas as pd
import evaluate

from underthesea import word_tokenize
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

# =========================
# CONFIG
# =========================

# TEST_PATH = "/kaggle/input/CHANGE_ME/test-00000-of-00001.parquet"

BASE_MODEL_NAME = "vinai/bartpho-word"
ADAPTER_DIR = "/kaggle/input/datasets/lhuythc/output-bartpho/checkpoints/bartpho_word_lora_correct"

OUTPUT_DIR = "/kaggle/working/official_test_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PRED_PATH = os.path.join(OUTPUT_DIR, "official_test_bartpho_word_lora_predictions.csv")
OUTPUT_EVAL_PATH = os.path.join(OUTPUT_DIR, "official_test_bartpho_word_lora_eval.csv")
OUTPUT_METRICS_PATH = os.path.join(OUTPUT_DIR, "official_test_bartpho_word_lora_metrics.csv")

MAX_INPUT_LENGTH = 1024
MAX_OUTPUT_LENGTH = 200
MIN_OUTPUT_LENGTH = 20
NUM_BEAMS = 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


# =========================
# READ TEST
# =========================

def read_any(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path.endswith(".csv"):
        return pd.read_csv(path)
    elif path.endswith(".json"):
        return pd.read_json(path)
    elif path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    else:
        raise ValueError(f"Unsupported file type: {path}")


def clean_test_df(df, text_col="article", summary_col="summary"):
    df = df.copy()

    if text_col not in df.columns:
        raise ValueError(f"Không tìm thấy cột {text_col}. Columns: {df.columns.tolist()}")

    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].str.len() > 50].reset_index(drop=True)

    if summary_col in df.columns:
        df[summary_col] = df[summary_col].astype(str).str.strip()
        df = df.rename(columns={text_col: "article", summary_col: "summary"})
        return df[["article", "summary"]].reset_index(drop=True)
    else:
        df = df.rename(columns={text_col: "article"})
        return df[["article"]].reset_index(drop=True)


raw_test_df = read_any(TEST_PATH)
official_test_df = clean_test_df(raw_test_df)

print("Official test:", official_test_df.shape)
print(official_test_df.columns)
display(official_test_df.head())


# =========================
# PREPROCESS
# =========================

def preprocess_for_bartpho_word(text):
    return word_tokenize(str(text), format="text")


# =========================
# GENERATE
# =========================

def generate_bartpho_word_lora(
    test_df,
    base_model_name,
    adapter_dir,
    output_path
):
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=False)

    print("Loading base model...")
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32

    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        base_model_name,
        torch_dtype=dtype
    )

    print("Loading LoRA adapter...")
    model = PeftModel.from_pretrained(base_model, adapter_dir)

    # merge adapter để inference nhanh hơn
    model = model.merge_and_unload()

    model.to(DEVICE)
    model.eval()

    results = []

    for i, row in test_df.iterrows():
        article = str(row["article"])
        source = preprocess_for_bartpho_word(article)

        inputs = tokenizer(
            source,
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            return_tensors="pt"
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        start = time.time()

        with torch.no_grad():
            output_ids = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=MAX_OUTPUT_LENGTH,
                min_length=MIN_OUTPUT_LENGTH,
                num_beams=NUM_BEAMS,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        elapsed = time.time() - start

        prediction = tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True
        ).strip()

        # Detokenize cho dễ đọc
        prediction_readable = prediction.replace("_", " ")

        item = {
            "id": i,
            "article": article,
            "prediction": prediction,
            "prediction_readable": prediction_readable,
            "inference_time_sec": elapsed,
            "base_model": base_model_name,
            "adapter": adapter_dir
        }

        if "summary" in test_df.columns:
            item["reference"] = str(row["summary"])

        results.append(item)

        if (i + 1) % 10 == 0:
            avg_time = sum(x["inference_time_sec"] for x in results) / len(results)
            print(f"Done {i + 1}/{len(test_df)} | avg = {avg_time:.2f}s/sample")

    out_df = pd.DataFrame(results)
    out_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("Saved:", output_path)
    print("Average time:", out_df["inference_time_sec"].mean())

    del model
    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    return out_df


pred_df = generate_bartpho_word_lora(
    test_df=official_test_df,
    base_model_name=BASE_MODEL_NAME,
    adapter_dir=ADAPTER_DIR,
    output_path=OUTPUT_PRED_PATH
)

display(pred_df.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.

,article,summary
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."
3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Du khách Anh Monisha Rajesh bắt đầu hành trình...
4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Cồn Én là một điểm đến du lịch nằm giữa sông T...


Loading tokenizer...


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading LoRA adapter...
Done 10/1344 | avg = 2.00s/sample
Done 20/1344 | avg = 1.72s/sample
Done 30/1344 | avg = 1.69s/sample
Done 40/1344 | avg = 1.61s/sample
Done 50/1344 | avg = 1.54s/sample
Done 60/1344 | avg = 1.54s/sample
Done 70/1344 | avg = 1.50s/sample
Done 80/1344 | avg = 1.47s/sample
Done 90/1344 | avg = 1.46s/sample
Done 100/1344 | avg = 1.44s/sample
Done 110/1344 | avg = 1.44s/sample
Done 120/1344 | avg = 1.41s/sample
Done 130/1344 | avg = 1.41s/sample
Done 140/1344 | avg = 1.39s/sample
Done 150/1344 | avg = 1.39s/sample
Done 160/1344 | avg = 1.39s/sample
Done 170/1344 | avg = 1.40s/sample
Done 180/1344 | avg = 1.40s/sample
Done 190/1344 | avg = 1.40s/sample
Done 200/1344 | avg = 1.40s/sample
Done 210/1344 | avg = 1.39s/sample
Done 220/1344 | avg = 1.39s/sample
Done 230/1344 | avg = 1.39s/sample
Done 240/1344 | avg = 1.40s/sample
Done 250/1344 | avg = 1.40s/sample
Done 260/1344 | avg = 1.39s/sample
Done 270/1344 | avg = 1.38s/sample
Done 280/1344 | avg = 1.37s/sample
Done 

,id,article,prediction,prediction_readable,inference_time_sec,base_model,adapter,reference
0,0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip vừa khai_trương văn_phòng mới tại TP H...,MayTrip vừa khai trương văn phòng mới tại TP H...,4.465089,vinai/bartpho-word,/kaggle/input/datasets/lhuythc/output-bartpho/...,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...","Đầu tháng 11 , dã_quỳ bung_nở trên các vạt núi...","Đầu tháng 11 , dã quỳ bung nở trên các vạt núi...",2.639297,vinai/bartpho-word,/kaggle/input/datasets/lhuythc/output-bartpho/...,Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay , Tam_Cốc đặc_biệt và hấp...","Mùa thu đông năm nay , Tam Cốc đặc biệt và hấp...",1.326898,vinai/bartpho-word,/kaggle/input/datasets/lhuythc/output-bartpho/...,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."
3,3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...","Monisha_Rajesh , nữ du_khách Anh , bước ra khỏ...","Monisha Rajesh , nữ du khách Anh , bước ra khỏ...",1.792823,vinai/bartpho-word,/kaggle/input/datasets/lhuythc/output-bartpho/...,Du khách Anh Monisha Rajesh bắt đầu hành trình...
4,4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...","Cồn_Én nằm giữa sông Tiền , thuộc xã Tấn_Mỹ , ...","Cồn Én nằm giữa sông Tiền , thuộc xã Tấn Mỹ , ...",2.594304,vinai/bartpho-word,/kaggle/input/datasets/lhuythc/output-bartpho/...,Cồn Én là một điểm đến du lịch nằm giữa sông T...


In [7]:
rouge = evaluate.load("rouge")

if "reference" in pred_df.columns:
    predictions = pred_df["prediction"].fillna("").astype(str).tolist()
    references = pred_df["reference"].fillna("").astype(str).tolist()

    scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=False
    )

    metrics = {
        "model": "bartpho_word_lora_official_test",
        "num_test_samples": len(pred_df),
        "rouge1": float(scores["rouge1"]),
        "rouge2": float(scores["rouge2"]),
        "rougeL": float(scores["rougeL"]),
        "rougeLsum": float(scores["rougeLsum"]),
        "avg_time_sec_per_sample": float(pred_df["inference_time_sec"].mean())
    }

    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(OUTPUT_METRICS_PATH, index=False, encoding="utf-8-sig")

    eval_df = pd.DataFrame()
    eval_df["id"] = pred_df["id"]
    eval_df["prediction"] = pred_df["prediction"]
    eval_df["prediction_readable"] = pred_df["prediction_readable"]
    eval_df["reference"] = pred_df["reference"]
    eval_df["source"] = pred_df["article"]

    eval_df.to_csv(OUTPUT_EVAL_PATH, index=False, encoding="utf-8-sig")

    print(metrics)
    display(metrics_df)
else:
    print("Official test không có reference/summary, chỉ xuất prediction để nộp.")

{'model': 'bartpho_word_lora_official_test', 'num_test_samples': 1344, 'rouge1': 0.7104759692548632, 'rouge2': 0.4358263395546329, 'rougeL': 0.4611264676762351, 'rougeLsum': 0.4632079825252684, 'avg_time_sec_per_sample': 1.3965602753063042}


,model,num_test_samples,rouge1,rouge2,rougeL,rougeLsum,avg_time_sec_per_sample
0,bartpho_word_lora_official_test,1344,0.710476,0.435826,0.461126,0.463208,1.39656
